In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [17]:
import os
import requests
import pandas as pd
from io import StringIO

# Load GitHub Token from environment variable (or store in .env)
token = os.getenv("GITHUB_TOKEN")

url = "https://raw.githubusercontent.com/SubharupBiswas/sih2025/main/crop_yield_dataset.csv"
headers = {"Authorization": f"token {token}"} if token else {}

# Load dataset locally if available, or fetch via URL
if os.path.exists("../datasets/crop_yield_dataset.csv"):
    df = pd.read_csv("../datasets/crop_yield_dataset.csv")
elif os.path.exists("ml/datasets/crop_yield_dataset.csv"):
    df = pd.read_csv("ml/datasets/crop_yield_dataset.csv")
elif os.path.exists("crop_yield_dataset.csv"):
    df = pd.read_csv("crop_yield_dataset.csv")
else:
    response = requests.get(url, headers=headers)
    df = pd.read_csv(StringIO(response.text))

print(df)   # (rows, columns)


             Date  Crop_Type Soil_Type  Soil_pH  Temperature   Humidity  \
0      2014-01-01      Wheat     Peaty     5.50     9.440599  80.000000   
1      2014-01-01       Corn     Loamy     6.50    20.052576  79.947424   
2      2014-01-01       Rice     Peaty     5.50    12.143099  80.000000   
3      2014-01-01     Barley     Sandy     6.75    19.751848  80.000000   
4      2014-01-01    Soybean     Peaty     5.50    16.110395  80.000000   
...           ...        ...       ...      ...          ...        ...   
36515  2023-12-31     Cotton      Clay     6.25    19.538555  80.000000   
36516  2023-12-31  Sugarcane     Peaty     5.50    21.068336  78.931664   
36517  2023-12-31     Tomato     Sandy     6.75     6.030148  80.000000   
36518  2023-12-31     Potato     Peaty     5.50    11.079561  80.000000   
36519  2023-12-31  Sunflower      Clay     6.25    11.455692  80.000000   

       Wind_Speed     N     P     K  Crop_Yield  Soil_Quality  
0       10.956707  60.5  45.0  31.5

In [18]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())

before_dupes = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after_dupes = df.shape[0]
print(f"\nDropped duplicates: {before_dupes - after_dupes}")

Shape: (36520, 12)
Columns: ['Date', 'Crop_Type', 'Soil_Type', 'Soil_pH', 'Temperature', 'Humidity', 'Wind_Speed', 'N', 'P', 'K', 'Crop_Yield', 'Soil_Quality']

Data types:
 Date             object
Crop_Type        object
Soil_Type        object
Soil_pH         float64
Temperature     float64
Humidity        float64
Wind_Speed      float64
N               float64
P               float64
K               float64
Crop_Yield      float64
Soil_Quality    float64
dtype: object

Missing values per column:
 Date            0
Crop_Type       0
Soil_Type       0
Soil_pH         0
Temperature     0
Humidity        0
Wind_Speed      0
N               0
P               0
K               0
Crop_Yield      0
Soil_Quality    0
dtype: int64

Dropped duplicates: 0


In [19]:
date_cols = [c for c in df.columns if "date" in c.lower()]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")
    df[f"{c}_Year"] = df[c].dt.year
    df[f"{c}_Month"] = df[c].dt.month
    df[f"{c}_DayOfYear"] = df[c].dt.dayofyear
    df[f"{c}_Week"] = df[c].dt.isocalendar().week.astype("Int64")
    df[f"{c}_Quarter"] = df[c].dt.quarter

In [20]:
possible_targets = ["Crop_Yield", "Yield", "yield", "target", "Target"]
target_col = next((c for c in possible_targets if c in df.columns), None)
if target_col is None:
    raise ValueError("Target column not found. Expected one of: " + ", ".join(possible_targets))

In [21]:
# 8) Split features (X) and target (y); drop raw date columns from features to avoid leakage
X = df.drop(columns=[target_col] + date_cols)
y = df[target_col]

In [7]:
# 9) Identify numeric and categorical feature columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['Soil_pH', 'Temperature', 'Humidity', 'Wind_Speed', 'N', 'P', 'K', 'Soil_Quality', 'Date_Year', 'Date_Month', 'Date_DayOfYear', 'Date_Week', 'Date_Quarter']
Categorical features: ['Crop_Type', 'Soil_Type']


In [8]:
# 10) Build preprocessing pipelines
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))  # Changed 'sparse' to 'sparse_output'
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
# 11) Fit and transform the dataset
X_processed = preprocess.fit_transform(X)

In [10]:
# 12) Recover feature names after transformation
num_feature_names = numeric_features
cat_feature_names = []
if len(categorical_features) > 0:
    cat_feature_names = list(
        preprocess.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
    )
final_feature_names = num_feature_names + cat_feature_names

In [11]:
# 13) Wrap into a processed DataFrame
X_processed_df = pd.DataFrame(X_processed, columns=final_feature_names, index=X.index)

In [12]:
# 14) Train-test split (stratify not used for regression)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed_df, y, test_size=0.2, random_state=42, shuffle=True
)

print("\nProcessed shapes:")
print("X_train:", X_train.shape, "X_test:", X_test.shape, "y_train:", y_train.shape, "y_test:", y_test.shape)


Processed shapes:
X_train: (29216, 28) X_test: (7304, 28) y_train: (29216,) y_test: (7304,)


In [13]:
# 15) Save processed datasets to disk
from pathlib import Path

# Create the directory if it doesn't exist
output_dir = Path("D:/SIH")
output_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
train_out = output_dir / "processed_train.csv"
test_out = output_dir / "processed_test.csv"
y_train_out = output_dir / "processed_y_train.csv"
y_test_out = output_dir / "processed_y_test.csv"

X_train.to_csv(train_out, index=False)
X_test.to_csv(test_out, index=False)
pd.DataFrame({"Crop_Yield": y_train}).to_csv(y_train_out, index=False)
pd.DataFrame({"Crop_Yield": y_test}).to_csv(y_test_out, index=False)

print("\nSaved:")
print(" -", train_out)
print(" -", test_out)
print(" -", y_train_out)
print(" -", y_test_out)


Saved:
 - D:\SIH\processed_train.csv
 - D:\SIH\processed_test.csv
 - D:\SIH\processed_y_train.csv
 - D:\SIH\processed_y_test.csv


In [14]:
from IPython.display import display

display(X_train.head(50))

,Soil_pH,Temperature,Humidity,Wind_Speed,N,P,K,Soil_Quality,Date_Year,Date_Month,...,Crop_Type_Soybean,Crop_Type_Sugarcane,Crop_Type_Sunflower,Crop_Type_Tomato,Crop_Type_Wheat,Soil_Type_Clay,Soil_Type_Loamy,Soil_Type_Peaty,Soil_Type_Saline,Soil_Type_Sandy
12353,-1.349796,0.116595,0.131409,-0.700875,-1.011433,-0.909364,-0.822031,-0.848257,-0.522207,-0.441796,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
13488,0.180264,-0.804628,0.848671,0.838785,-1.011433,-1.476722,-1.759052,-0.109208,-0.522207,0.718117,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
35370,-0.125748,0.235788,-0.025703,-0.482344,1.009405,0.792713,0.349245,1.307695,1.567002,0.718117,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
16486,0.180264,0.856321,-0.843642,-1.301619,-0.092870,-0.568948,-0.704904,0.335634,-0.174005,0.138161,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
31291,-0.431760,1.927706,-2.255861,0.754641,0.550124,0.225354,-0.236393,0.632195,1.218801,0.138161,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
12932,-0.125748,1.298666,-1.426707,-0.729154,2.295393,2.154375,1.520522,1.985549,-0.522207,0.138161,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5211,-0.125748,1.487551,-1.675681,-0.250260,1.652399,1.473544,0.934884,1.646622,-1.218610,-0.151818,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
15644,-1.349796,-0.455568,0.848671,-0.203795,-1.516642,-0.909364,-0.412084,-0.867086,-0.174005,-0.731775,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
800,-0.431760,0.761797,-0.719048,-0.825598,-0.001014,-0.342005,-0.704904,0.385061,-1.566812,-1.021753,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
8978,-1.349796,1.428935,-1.598419,-1.354576,-0.506223,-0.909364,-1.231978,-0.829428,-0.870409,-0.151818,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Train baseline Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predictions
y_pred = lr.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression RMSE:", rmse)
print("Linear Regression R²:", r2)


Linear Regression RMSE: 15.340471093777936
Linear Regression R²: 0.647090292794974
